# 🎨 Diffusion Models Workshop: Introduction

**Introduction to AI Image Generation**

*Adapted from the [Hugging Face Diffusion Models Course](https://github.com/huggingface/diffusion-models-class)*

---

### What you'll learn today:
1. **Load** a pretrained diffusion model and understand what it does
2. **Generate** images from text prompts
3. **Visualize** how diffusion models transform "noise" into images, step by step

### Before we start:
- Make sure you're running this notebook on **Google Colab** with a **GPU runtime**
- Go to **Runtime → Change runtime type → T4 GPU**
- You'll need a free [Hugging Face account](https://huggingface.co/join) (create one now if you don't have one)

### Setting up your Hugging Face account and access token:
1. Go to [huggingface.co/join](https://huggingface.co/join) and create a free account (or log in if you already have one)
2. Once logged in, go to your **token settings**: [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
3. Click **"Create new token"** → give it a name (e.g., `workshop`) → select **"Read"** access → click **"Create"**
4. Copy the token (it starts with `hf_...`) — you'll paste it in the next cell
5. Run the login cell below and paste your token when prompted

In [ ]:
!pip install huggingface_hub

# Log in to Hugging Face — paste your access token when prompted
from huggingface_hub import notebook_login
notebook_login()

---
## 🔧 Step 0: Check our GPU and install libraries

Let's make sure we have a GPU available and install everything we need.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

!pip install -q diffusers transformers accelerate safetensors gradio matplotlib

print("✅ All libraries installed!")

Now let's make sure that everything installed correctly

In [ ]:
# Check that installation worked correctly
import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")


---
## ⚙️ Section 1: Loading a Pretrained Diffusion Model

### What is a diffusion model?

A diffusion model is a type of neural network that generates images. We'll explore exactly how this works in Section 2, but first let's load a model and generate our first image.

### The model we'll use

We'll load **Stable Diffusion 1.5** — one of the most widely-used diffusion models.

The model has two main components:
- **Text Encoder**: Converts your text prompt into numbers the model understands.
- **UNet**: A neural network takes these numbers and generates an image.

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

# Load the model (this downloads ~4GB the first time)
model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,        # Use half precision to save memory
    safety_checker=None,               # Disable for speed in workshop setting
    requires_safety_checker=False
)
pipe = pipe.to("cuda")

# Enable memory-efficient attention (important for T4 GPU!)
pipe.enable_attention_slicing()

print("✅ Model loaded successfully!")

### 🧪 Quick test: Generate your first image!

Let's generate a simple image to make sure everything works.


In [ ]:
# Generate a test image
prompt = "a painting in a sunny studio of the numbers '6' and '7'"

image = pipe(
    prompt,
    num_inference_steps=25,   # Number of denoising steps
    guidance_scale=7.5        # How closely to follow the prompt
).images[0]

display(image)
print(f"Image size: {image.size}")


---
## 🔬 Section 2: Visualizing the Diffusion Process

No we're going to explore how diffusion models work, by watching it generate an image, step by step.

### The forward process (adding noise)

Imagine you have a photograph, and you gradually add noise to it — like an old TV losing signal. Eventually, the image becomes pure random noise. Let's watch this process for the image that we just generated:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from diffusers import DDPMScheduler
import torch

# --- Part A: Visualize the FORWARD process (adding noise) ---
# Use the image we just generated above!
sample_img = image.resize((256, 256))

# Set up the noise scheduler
noise_scheduler = DDPMScheduler(num_train_timesteps=1000)

# Convert image to tensor
img_tensor = torch.tensor(np.array(sample_img)).float() / 255.0
img_tensor = (img_tensor * 2 - 1)  # Scale to [-1, 1]
img_tensor = img_tensor.permute(2, 0, 1).unsqueeze(0)  # [B, C, H, W]

# Show the image at different noise levels
timesteps = [0, 50, 100, 200, 400, 600, 800, 999]
fig, axes = plt.subplots(1, len(timesteps), figsize=(20, 3))
fig.suptitle("Forward Process: Gradually Adding Noise to an Image", fontsize=14, fontweight='bold')

for idx, t in enumerate(timesteps):
    noise = torch.randn_like(img_tensor)
    t_tensor = torch.tensor([t])
    noisy = noise_scheduler.add_noise(img_tensor, noise, t_tensor)

    # Convert back to displayable image
    noisy_img = (noisy.squeeze().permute(1, 2, 0).numpy() + 1) / 2
    noisy_img = np.clip(noisy_img, 0, 1)

    axes[idx].imshow(noisy_img)
    axes[idx].set_title(f"t = {t}", fontsize=10)
    axes[idx].axis("off")

plt.tight_layout()
plt.show()

### The reverse process (removing noise)
**A diffusion model learns to reverse the noising process**: given pure noise, it gradually "denoises" it into a coherent image. If we also tell the model *what* we want (via a text prompt), it can guide the denoising toward a specific image. Let's use the same prompt to perform denoising:

In [ ]:
# --- Part B: Visualize the REVERSE process (denoising step by step) ---
# We'll capture intermediate images during generation

intermediate_images = []
capture_steps = set()

def callback_fn(pipe, step_index, timestep, callback_kwargs):
    """Capture latents at specific steps during denoising."""
    if step_index in capture_steps:
        latents = callback_kwargs["latents"]
        with torch.no_grad():
            # Decode latents to image
            image = pipe.vae.decode(latents / pipe.vae.config.scaling_factor, return_dict=False)[0]
            image = (image / 2 + 0.5).clamp(0, 1)
            image = image.cpu().permute(0, 2, 3, 1).float().numpy()[0]
            intermediate_images.append((step_index, image))
    return callback_kwargs

# Set which steps to capture
total_steps = 30
# Capture more at the start (where big changes happen) and space out later
capture_indices = [0, 1, 2, 3, 5, 8, 12, 17, 22, total_steps - 1]
capture_steps = set(capture_indices)

# Generate with callback
intermediate_images = []
generator = torch.Generator("cuda").manual_seed(42)

with torch.autocast("cuda"):
    final_image = pipe(
        prompt,
        num_inference_steps=total_steps,
        guidance_scale=7.5,
        generator=generator,
        callback_on_step_end=callback_fn,
    ).images[0]

# Plot the denoising progression
n_images = len(intermediate_images)
fig, axes = plt.subplots(1, n_images, figsize=(2.5 * n_images, 3))
fig.suptitle(f'Reverse Process: Denoising "{prompt}"', fontsize=12, fontweight='bold')

for idx, (step, img) in enumerate(intermediate_images):
    axes[idx].imshow(np.clip(img, 0, 1))
    axes[idx].set_title(f"Step {step}", fontsize=9)
    axes[idx].axis("off")

plt.tight_layout()
plt.show()

### What do you notice?

Notice how the result looks different than what we started with!

### Why does this happen?

The model sees millions of images from the internet, each paired with a text caption. For every image, the model is "trained" like this:

1. **Adds noise** to the image
2. **Show the noisy image to the model** and ask it: *"what was the image I started with?"*
3. **Compare the model's guess** to the original image

This is repeated *millions* of times. Over time, the model gets very good at looking at a noisy image and predicting the picture hiding underneath the noise. *But* the model might see the same caption multiple times along with different images. When this happens, the model doesn't know for sure which image is "right", and picks a random one.

In [ ]:

# --- Visualizing WHY the same prompt gives different images ---
# We'll generate 3 different "dog" images, show how each gets noised
# during training, and how the model learns to predict the original.
# Then we show: same prompt → different outputs (because the model
# learned from MANY different dogs, all captioned "a photo of a dog").

from matplotlib.patches import ConnectionPatch

# ── Step 1: Generate 3 different dog images with different seeds ──
dog_prompt = "a photo of a dog"
dog_images = []
seeds = [11, 84, 2024]

print("Generating 3 different dog images from the same prompt...")
for seed in seeds:
    generator = torch.Generator("cuda").manual_seed(seed)
    with torch.autocast("cuda"):
        img = pipe(dog_prompt, num_inference_steps=25, guidance_scale=7.5,
                   generator=generator).images[0]
    dog_images.append(img.resize((256, 256)))
    print(f"  ✅ Dog {len(dog_images)} (seed={seed})")

# ── Step 2: Build the training-process diagram ──
# Each row: [Original dog] --noise→ [Noisy version] --model predicts→ [Original dog]

noise_timestep = 400  # moderate noise level

fig, axes = plt.subplots(3, 3, figsize=(14, 12))
fig.suptitle('During Training: Same Caption, Different Images\n'
             'The model sees many different dogs, all labeled "a photo of a dog"',
             fontsize=14, fontweight='bold', y=0.98)

for row, dog_img in enumerate(dog_images):
    # Convert to tensor and add noise
    img_tensor = torch.tensor(np.array(dog_img)).float() / 255.0
    img_tensor = (img_tensor * 2 - 1).permute(2, 0, 1).unsqueeze(0)
    noise = torch.randn_like(img_tensor)
    t_tensor = torch.tensor([noise_timestep])
    noisy = noise_scheduler.add_noise(img_tensor, noise, t_tensor)
    noisy_img = np.clip((noisy.squeeze().permute(1, 2, 0).numpy() + 1) / 2, 0, 1)

    # Column 0: Original image (training data)
    axes[row, 0].imshow(np.array(dog_img))
    axes[row, 0].set_title(f"Training image {row+1}", fontsize=11)
    axes[row, 0].axis("off")

    # Column 1: Noisy version
    axes[row, 1].imshow(noisy_img)
    axes[row, 1].set_title("Add noise", fontsize=11)
    axes[row, 1].axis("off")

    # Column 2: Model's prediction
    axes[row, 2].imshow(np.array(dog_img))
    axes[row, 2].set_title('Model predicts: "original"', fontsize=11)
    axes[row, 2].axis("off")

    # Draw arrow: original (col0) → noisy (col1)
    arrow_fwd = ConnectionPatch(
        xyA=(1.0, 0.5), coordsA=axes[row, 0].transAxes,
        xyB=(0.0, 0.5), coordsB=axes[row, 1].transAxes,
        arrowstyle='->', mutation_scale=20,
        color='#3498db', linewidth=2.5,
    )
    fig.add_artist(arrow_fwd)

    # Draw arrow: noisy (col1) → model prediction (col2)
    arrow_rev = ConnectionPatch(
        xyA=(1.0, 0.5), coordsA=axes[row, 1].transAxes,
        xyB=(0.0, 0.5), coordsB=axes[row, 2].transAxes,
        arrowstyle='->', mutation_scale=20,
        color='#e74c3c', linewidth=2.5,
    )
    fig.add_artist(arrow_rev)

plt.tight_layout(rect=[0, 0.08, 1, 0.94])

# Add centered caption label under each row using figure coordinates
for row in range(3):
    bbox_left = axes[row, 0].get_position()
    bbox_right = axes[row, 2].get_position()
    center_x = (bbox_left.x0 + bbox_right.x1) / 2
    label_y = bbox_left.y0 - 0.015
    fig.text(center_x, label_y, f'caption: "{dog_prompt}"',
             ha='center', va='top', fontsize=10, fontstyle='italic', color='gray')

plt.show()

# ── Step 3: Show that the SAME prompt generates DIFFERENT images ──
fig2, axes2 = plt.subplots(1, 3, figsize=(14, 4.5))
fig2.suptitle('At Generation Time: Same Prompt → Different Results\n'
              f'Prompt: "{dog_prompt}"   (different random seeds)',
              fontsize=14, fontweight='bold')

for i, dog_img in enumerate(dog_images):
    axes2[i].imshow(np.array(dog_img))
    axes2[i].set_title(f"seed = {seeds[i]}", fontsize=11)
    axes2[i].axis("off")
    for spine in axes2[i].spines.values():
        spine.set_visible(True)
        spine.set_color('#e74c3c')
        spine.set_linewidth(2)

plt.tight_layout()
fig2.text(0.5, -0.02,
          'Because the model learned from many different dogs with the same caption,\n'
          'it can generate a different dog each time — depending on the starting noise!',
          ha='center', fontsize=11, fontstyle='italic', color='#555')
plt.show()

print("💡 Key insight: The model doesn't memorize ONE dog for the caption 'a photo of a dog'.")
print("   It learns the DISTRIBUTION of all dogs it has seen, and samples a different one each time!")

---
## ✍️ Section 3: Image Generation with an Interactive App

Now let's build an interactive app!

### Key parameters explained:
| Parameter | What it does |
|-----------|-------------|
| **Prompt** | Text description of the image you want |
| **Negative prompt** | Things you want to *avoid* in the image |


Run the code in the cell below

In [ ]:
import gradio as gr

def generate_image(prompt, negative_prompt, num_steps=25, guidance_scale=7.5, seed=-1, width=512, height=512):
    """Generate an image from a text prompt."""
    generator = torch.Generator("cuda").manual_seed(seed)

    with torch.autocast("cuda"):
        result = pipe(
            prompt=prompt,
            negative_prompt=negative_prompt if negative_prompt else None,
            num_inference_steps=num_steps,
            guidance_scale=guidance_scale,
            generator=generator,
            width=width,
            height=height,
        )

    return result.images[0], f"Seed: {seed}"

# Build the Gradio interface
demo = gr.Interface(
    fn=generate_image,
    inputs=[
        gr.Textbox(
            label="✏️ Prompt",
            placeholder="Describe the image you want to create...",
            value="a magical forest with glowing mushrooms, fantasy art, detailed, vibrant colors",
            lines=2
        ),
        gr.Textbox(
            label="🚫 Negative Prompt (things to avoid)",
            placeholder="blurry, low quality, distorted...",
            value="blurry, low quality, distorted, ugly",
            lines=1
        ),
    ],
    outputs=[
        gr.Image(label="Generated Image", type="pil"),
        gr.Textbox(label="Info"),
    ],
    title="🎨 Text-to-Image Generator",
    description="Type a description and watch the diffusion model create an image! Try different prompts and settings.",
)

demo.launch(debug=True, share=True)

---
## 🎓 Wrap-Up: What Did We Learn?

### Key concepts:
1. **Diffusion models** generate images by learning to reverse a noise-adding process
2. **Text** lets us guide image generation with natural language
3. **Negative prompts** give us more controls over the diffusion process, by telling the model what *not to* include.

### The bigger picture:
- These models are trained on millions of image-text pairs scraped from the internet
- This raises important questions about **copyright**, **bias**, and **consent**
- The open-source nature of these models means anyone can inspect, modify, and improve them


### Going further:
- 📚 [Hugging Face Diffusion Models Course](https://huggingface.co/learn/diffusion-course/) — the full course we adapted today
- 📚 [Stable Diffusion Deep Dive](https://fastai.github.io/diffusion-nbs/) — from fast.ai

### 🧠 Discussion questions:
1. What are the implications of being able to generate any image from text?
2. What are the ethical boundaries of generating images of real people?
3. How might diffusion models change creative industries?
4. What biases might exist in the training data, and how would they show up in generated images?
5. Should there be regulations around AI image generation? What kind?